In [1]:
# Core
import os
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt
os.chdir('/home/labs/davidgo/Collaboration')

In [2]:
jaspar_metadata = pd.read_csv('USEFUL_DATASETS/Sequence/Motifs_and_TFBS/JASPAR2024/JASPAR2024_CORE_vertebrates_non-redundant_meta.tsv', 
                     header=0,sep = '\t')
jaspar_metadata = jaspar_metadata.rename(columns={'Unnamed: 0': 'Matrix_ID'})

# Filtering settings

In [8]:
main = True
experiment_type = False
active_in_cell_type = 'chondrocyte' # OPTIONS: osteoblast, chondrocyte, npc, None

## Filter for main matrices based on Omer's decision

In [9]:

print(len(jaspar_metadata))
if main:
    filtered_jaspar_metadata = jaspar_metadata[jaspar_metadata['main']]
print(len(filtered_jaspar_metadata))

879
828


## Filter for TFs which were measured using SELEX (any kind) or PBM

In [10]:
print(len(filtered_jaspar_metadata))
if experiment_type:
    mask = jaspar_metadata['experiment_type'].isin(['HT-SELEX','CAP-SELEX','PBM','SELEX','NCAP-SELEX'])
    filtered_jaspar_metadata = filtered_jaspar_metadata[mask]
print(len(filtered_jaspar_metadata))



828
828


In [11]:
gene_annotation_table = pd.read_csv('/home/labs/davidgo/nadavmi/usefull/Human.GRCh38.p13.annot.tsv', 
                     header=0,sep = '\t',usecols=[0,1])

## Extract expression in articular chondrocytes

In [12]:
articular_cartilage_TPM = pd.read_csv('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/GSE114007_human_control_and_osteoarithritis_cartilage/GSE114007_norm_counts_TPM_GRCh38.p13_NCBI.tsv', 
                     header=0,sep = '\t')

articular_cartilage_TPM = articular_cartilage_TPM.iloc[:,0:9]
articular_cartilage_TPM['articular_cartilage_mean'] = articular_cartilage_TPM.iloc[:, 1:].mean(axis=1)
articular_cartilage_TPM = articular_cartilage_TPM[['GeneID','articular_cartilage_mean']]


articular_cartilage_TPM = pd.merge(articular_cartilage_TPM, gene_annotation_table, on='GeneID', how='outer') 
articular_cartilage_TPM = articular_cartilage_TPM.set_index('Symbol')
articular_cartilage_TPM = articular_cartilage_TPM[['articular_cartilage_mean']]
articular_cartilage_TPM = articular_cartilage_TPM.groupby('Symbol', as_index=True).mean() # Group by gene name and average duplicates


## Extract expression in osteoblasts

In [13]:
osteoblast_TPM = pd.read_csv('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Bone/Human/RNA-seq/GSE57925_norm_counts_TPM_GRCh38.p13_NCBI.tsv', 
                     header=0,sep = '\t')
osteoblast_TPM = pd.merge(osteoblast_TPM,gene_annotation_table,on = 'GeneID')
expression = osteoblast_TPM.iloc[:, 1:4].mean(axis=1)
osteoblast_TPM = pd.DataFrame({'Symbol': osteoblast_TPM['Symbol'],'osteoblast_mean': expression})
osteoblast_TPM = osteoblast_TPM.groupby('Symbol', as_index=True).mean() # Group by gene name and average duplicates


## Extract expression in NPCs

In [14]:
npcs_TPM = pd.read_excel('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Other tissues/NPCs/GSE84712_TPM_all_samples.xlsx', 
                 header=0, usecols = "A:AD")

npcs_TPM['expression'] = npcs_TPM.iloc[:, 3:].mean(axis=1)
npcs_TPM = npcs_TPM[['gene','expression']]
npcs_TPM.rename(columns={'gene': 'Symbol'}, inplace=True)
npcs_TPM = pd.DataFrame({'Symbol': npcs_TPM['Symbol'],'npc_mean': npcs_TPM['expression']})
npcs_TPM['Symbol'] = npcs_TPM['Symbol'].astype(str)
npcs_TPM = npcs_TPM.groupby('Symbol', as_index=True).mean() # Group by gene name and average duplicates



In [15]:
TPM_values = pd.concat([articular_cartilage_TPM, osteoblast_TPM, npcs_TPM], axis=1, join='outer')

chondrocyte_active_genes = TPM_values[TPM_values['articular_cartilage_mean'] > 1].index.tolist()
osteoblast_active_genes = TPM_values[TPM_values['osteoblast_mean'] > 1].index.tolist()
npc_active_genes = TPM_values[TPM_values['npc_mean'] > 1].index.tolist()


## Filter based on cell type

In [16]:
print(len(filtered_jaspar_metadata))
if active_in_cell_type == 'chondrocyte':
    print('filtering for genes active in chondrocytes')
    filtered_jaspar_metadata = filtered_jaspar_metadata[filtered_jaspar_metadata['TF'].isin(chondrocyte_active_genes)]

elif active_in_cell_type == 'osteoblast':
    print('filtering for genes active in osteoblasts')
    filtered_jaspar_metadata = filtered_jaspar_metadata[filtered_jaspar_metadata['TF'].isin(osteoblast_active_genes)]

elif active_in_cell_type == 'npc':
    print('filtering for genes active in NPCs')
    filtered_jaspar_metadata = filtered_jaspar_metadata[filtered_jaspar_metadata['TF'].isin(npc_active_genes)]

print(len(filtered_jaspar_metadata))


828
filtering for genes active in chondrocytes
371


## Define set of TFs to keep and save new JASPAR file

In [17]:
MAtrix_IDs_to_keep = filtered_jaspar_metadata['Matrix_ID'].tolist()

## Load original MEME file to filter

In [18]:

# Read the full file content
with open('USEFUL_DATASETS/Sequence/Motifs_and_TFBS/JASPAR2024/JASPAR2024_CORE_vertebrates_non-redundant_pfms_meme.meme', "r") as f:
    content = f.read()

# Extract the header (everything before the first MOTIF)
header_match = re.search(r"^(.*?)\n(?=MOTIF)", content, flags=re.DOTALL)
header = header_match.group(1) if header_match else ""

# Find all MOTIF blocks (from MOTIF to next MOTIF or end of file)
motif_blocks = re.findall(r"(MOTIF\s+(MA\d+\.\d+).*?)(?=MOTIF\s+MA\d+\.\d+|\Z)", content, flags=re.DOTALL)

# Filter blocks based on ID
filtered_blocks = [full_block for full_block, motif_id in motif_blocks if motif_id in MAtrix_IDs_to_keep]




# Write filtered MEME file

In [19]:
# Write result to file
with open(f"humanMPRA/TF_analysis/final/intermediate_files/filtered_MEME_files/filtered_{active_in_cell_type}_JASPAR2024_CORE_vertebrates_non-redundant_pfms_meme.meme", "w") as out:
    out.write(header.strip() + "\n\n" + "".join(filtered_blocks))